# N136 · 线段树与区间聚合

**目标：** 把结合律与单位元落实为可组合节点。

**先修：** N031, N063, N134。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 建树；点更新；半开区间；幺半群；结合不等于交换；非空与空查询。

## 从问题到算法

线段树只要求合并运算满足结合律并有单位元，不要求交换律。求和、最小值、字符串拼接都符合这一接口。区间查询从两端收集块时，左侧按从左到右追加，右侧必须向前插入；否则求和测试会通过，但字符串拼接会反序。

## 最小实现

**本章接口：** `SegmentTree(values, combine, identity)`、`SegmentTree.set(i, value)`、`SegmentTree.query(left, right)`

In [1]:
class SegmentTree:
    def __init__(self,values,combine,identity):
        self.n=len(values); self.combine=combine; self.identity=identity; self.size=1
        while self.size<self.n: self.size*=2
        self.tree=[identity]*(2*self.size)
        self.tree[self.size:self.size+self.n]=list(values)
        for i in range(self.size-1,0,-1): self.tree[i]=combine(self.tree[2*i],self.tree[2*i+1])
    def set(self,i,value):
        if not 0<=i<self.n: raise IndexError('point outside tree')
        i+=self.size; self.tree[i]=value
        while i>1:
            i//=2; self.tree[i]=self.combine(self.tree[2*i],self.tree[2*i+1])
    def query(self,left,right):
        if not 0<=left<=right<=self.n: raise IndexError('invalid half-open range')
        left+=self.size; right+=self.size; a=b=self.identity
        while left<right:
            if left&1: a=self.combine(a,self.tree[left]); left+=1
            if right&1: right-=1; b=self.combine(self.tree[right],b)
            left//=2; right//=2
        return self.combine(a,b)

## 正确性、前提与复杂度

每个节点保持其区间按原顺序的聚合。查询把区间拆成不重叠有序块；结合律允许改变括号，不能允许改变顺序。左右双累加器分别维护已取前缀与后缀，最终combine(a,b)恢复整体顺序。

**代价：** 建树O(n)次combine，更新与查询O(log n)次combine，空间O(n)。这些上界以combine为O(1)为前提；字符串拼接自身复制字符，真实字节代价不能忽略。identity应不可变，combine应无副作用。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tree=SegmentTree(list('abcdef'),lambda a,b:a+b,'')
show_table(['半开区间','聚合字符串'],[((l,r),tree.query(l,r)) for l,r in [(0,6),(1,5),(2,2),(5,6)]])
show_table(['内部节点','节点聚合'],list(enumerate(tree.tree[1:],1)))

半开区间,聚合字符串
"(0, 6)",abcdef
"(1, 5)",bcde
"(2, 2)",
"(5, 6)",f


内部节点,节点聚合
1,abcdef
2,abcd
3,ef
4,ab
5,cd
6,ef
7,
8,a
9,b
10,c


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
import operator
from random import Random
rng=Random(136)
for n in range(1,25):
    a=[rng.randrange(-9,10) for _ in range(n)]; tree=SegmentTree(a,operator.add,0)
    letters=list('x'*n); text=SegmentTree(letters,operator.add,'')
    for _ in range(70):
        i=rng.randrange(n); a[i]=rng.randrange(-9,10); tree.set(i,a[i]); letters[i]=rng.choice('abc'); text.set(i,letters[i])
        l=rng.randrange(n+1); r=rng.randrange(l,n+1)
        assert tree.query(l,r)==sum(a[l:r])
        assert text.query(l,r)==''.join(letters[l:r])
assert SegmentTree([],operator.add,0).query(0,0)==0
print('N136: 所有本章断言通过')

N136: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 同时支持sum/min/字符串拼接。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释左右累积顺序对非交换操作的重要性。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 307. Range Sum Query - Mutable（canonical）
- 2286. Booking Concert Tickets in Groups（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。